# Feature scaling

Age is in tens and salary is in tens of thousands. Models that measure **distance**, like K-NN, let the bigger numbers win. This notebook shows the problem and the fix: `StandardScaler`.

This notebook goes with the **Feature scaling** lesson on [ML Visual](https://mohammad-qt.github.io/ML_DL_Vis/). Run each cell in order with **Shift + Enter**. Nothing to install: Colab already has every library used here.

## 1. Load the data

`customers.csv` is a small made-up table of 80 shoppers: their **country**, **age** and **salary**, and whether they **purchased** (1) or not (0). It was generated for this course, so it is free to use. A few ages and salaries are left blank on purpose.

In [ ]:
import numpy as np
import pandas as pd

# The data lives in this project's GitHub repository, so there is nothing to upload.
DATA_URL = "https://raw.githubusercontent.com/Mohammad-QT/ML_DL_Vis/main/datasets/customers.csv"
dataset = pd.read_csv(DATA_URL)
dataset.head()

## 2. Split

We drop the rows with blanks, then hold out a test set. The scaler must only learn from the **training** rows.

In [ ]:
from sklearn.model_selection import train_test_split

data = dataset.dropna()
X = data[['age', 'salary']].values
y = data['purchased'].values
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=0)

print('Spread (standard deviation) of each column in the training set:')
print('age   ', X_train[:, 0].std().round(1))
print('salary', X_train[:, 1].std().round(1))

## 3. K-NN without scaling

Salary's spread is about a thousand times bigger than age's, so it decides almost every distance. Age hardly counts.

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

raw = KNeighborsClassifier(n_neighbors=5).fit(X_train, y_train)
print('Test accuracy without scaling:', round(raw.score(X_test, y_test), 3))

## 4. Scale the features

`StandardScaler` subtracts each column's mean and divides by its standard deviation. `fit_transform` learns those two numbers from the training rows and applies them. `transform` reuses the **same** numbers on the test rows.

In [ ]:
from sklearn.preprocessing import StandardScaler

sc = StandardScaler()
X_train_s = sc.fit_transform(X_train)
X_test_s = sc.transform(X_test)

print('Training columns now have mean', X_train_s.mean(axis=0).round(2), 'and spread', X_train_s.std(axis=0).round(2))

## 5. K-NN with scaling

Now age and salary both count when we measure distance.

In [ ]:
scaled = KNeighborsClassifier(n_neighbors=5).fit(X_train_s, y_train)
print('Test accuracy with scaling:   ', round(scaled.score(X_test_s, y_test), 3))
print('Test accuracy without scaling:', round(raw.score(X_test, y_test), 3))

## 6. See who the nearest neighbour is

Pick one test person and ask for their single nearest training neighbour, in raw units and in scaled units. This is what the lesson's picture shows.

In [ ]:
person = 0
raw_i = raw.kneighbors(X_test[[person]], n_neighbors=1, return_distance=False)[0][0]
scaled_i = scaled.kneighbors(X_test_s[[person]], n_neighbors=1, return_distance=False)[0][0]

print('Person:                ', X_test[person])
print('Nearest in raw units:  ', X_train[raw_i])
print('Nearest in scaled units:', X_train[scaled_i])

## Try it yourself

- Change `person` to other numbers. How often is the nearest neighbour different?
- Replace `KNeighborsClassifier` with `DecisionTreeClassifier` and compare the scores with and without scaling. Why does a tree not care?